# 04b · Five-Algorithm Predictive Benchmark — Phase 4.5

**Task:** Predict `gpu_hours_requested` per job before it runs.

**Why these five algorithms:**

1. **LinearRegression** — baseline; establishes the floor. Simple linear relationship between features and GPU-hours. No hyperparameters.
2. **RandomForest** — bagging ensemble; robust to outliers, handles mixed feature types, no scaling needed. Good benchmark for tree-based methods.
3. **XGBoost** — boosting ensemble; state-of-art on tabular data, handles missing values, fast inference via `hist` tree method.
4. **LightGBM** — histogram-based boosting; faster than XGBoost on large datasets (40M rows), good with high-cardinality categoricals via target encoding.
5. **LSTM** — sequence model; included to test whether temporal structure helps. **Note:** without a true time series (no start timestamps), this reduces to a 2-layer MLP with `seq_len=1`. Its performance reflects deep learning on tabular data, not sequence modeling.

**Data:** `data/interim/job_summary_canonical.parquet` — 40M rows, job-level features only.  
**Split:** Stratified random 70/15/15% (no temporal split possible — see paper §6.1).  
**Prerequisites:** `make model` (full, ~90 min) or `make model FAST=1` (~5 min on 10% sample).

In [ ]:
import sys
print(sys.executable)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from IPython.display import display, Markdown

from src.config import FIG_DIR, TABLE_DIR, LOG_DIR, PROCESSED_DIR
from src.viz.style import set_ieee_style

set_ieee_style()
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

_CMP = TABLE_DIR / 'model_comparison.csv'
_JT  = TABLE_DIR / 'model_comparison_per_jobtype.csv'
_FI  = TABLE_DIR / 'feature_importance.csv'
_RAT = LOG_DIR / 'model_selection_rationale.md'

if not _CMP.exists():
    print('Run `make model` or `make model FAST=1` first.')
else:
    cmp_df = pd.read_csv(_CMP)
    print('Ready. Algorithms:', cmp_df['algorithm'].tolist())

## Cell 2 — Styled comparison table

In [ ]:
cmp_df = pd.read_csv(_CMP)

def fmt_time(s):
    if pd.isna(s): return '—'
    if s < 60: return f'{s:.0f}s'
    return f'{s/60:.1f}min'

cmp_display = cmp_df.copy()
cmp_display['train_time'] = cmp_display['train_time_s'].apply(fmt_time)
cmp_display['latency']    = cmp_display['latency_ms_10k'].apply(lambda x: f'{x:.1f}ms' if not pd.isna(x) else '—')
cols = ['algorithm', 'mae', 'rmse', 'mape', 'r2', 'pinball_90', 'train_time', 'latency']

numeric_cols = ['mae', 'rmse', 'mape', 'r2', 'pinball_90']

def highlight(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in numeric_cols:
        if col not in df.columns: continue
        if col == 'r2':
            best_idx = df[col].idxmax()
            worst_idx = df[col].idxmin()
        else:
            best_idx = df[col].idxmin()
            worst_idx = df[col].idxmax()
        styles.loc[best_idx, col]  = 'background-color: #c6efce; color: #276221'
        styles.loc[worst_idx, col] = 'background-color: #ffc7ce; color: #9c0006'
    return styles

styled = (cmp_display[cols]
    .style
    .apply(highlight, axis=None)
    .format({c: '{:.4f}' for c in numeric_cols})
    .set_caption('Five-algorithm benchmark — test set results (green = best, red = worst)')
)
display(styled)

## Cell 3 — Figure: grouped bar chart (MAE, RMSE, R²)

In [ ]:
algos = cmp_df['algorithm'].tolist()
metrics = ['mae', 'rmse', 'r2']
labels  = ['MAE', 'RMSE', 'R²']

x = np.arange(len(algos))
width = 0.22
colors = ['#0072B2', '#E69F00', '#009E73']

fig, ax = plt.subplots(figsize=(7.0, 3.0))
for i, (metric, label, color) in enumerate(zip(metrics, labels, colors)):
    vals = cmp_df[metric].values
    bars = ax.bar(x + (i - 1) * width, vals, width, label=label, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(algos, rotation=15, ha='right', fontsize=7)
ax.set_ylabel('Score')
ax.set_title('Algorithm comparison: MAE, RMSE, R² on test set')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_model_comparison_bar.pdf')
plt.show()
print('Saved: fig8_model_comparison_bar.pdf')

## Cell 4 — Figure: predicted vs actual scatter + residuals of winner

In [ ]:
# Requires fitted models in memory — typically run after make model
# This cell shows a skeleton; actual models need to be loaded from disk.
from src.modeling.choose import load_production_model
from src.modeling.split import load_splits
from src.modeling.features import build_features

try:
    prod_model = load_production_model()
    feat_df = build_features()  # loads from cache
    _, _, test_df = load_splits(feat_df)

    feat_cols = [c for c in test_df.columns if c.startswith('feat_')]
    X_test = test_df[feat_cols].values.astype('float32')
    y_test = test_df['y'].values

    # Sample 10,000 rows for speed
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X_test), min(10_000, len(X_test)), replace=False)
    X_s, y_s = X_test[idx], y_test[idx]
    y_pred = prod_model.predict(X_s)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 3.0))
    lim = max(y_s.max(), y_pred.max())
    ax1.scatter(y_s, y_pred, s=2, alpha=0.3, color='#0072B2')
    ax1.plot([0, lim], [0, lim], 'r--', lw=1, label='y=x')
    ax1.set_xlabel('Actual GPU-hours')
    ax1.set_ylabel('Predicted GPU-hours')
    ax1.set_title(f'Predicted vs Actual ({prod_model.name})')
    ax1.legend()

    residuals = y_pred - y_s
    ax2.hist(residuals, bins=50, color='#E69F00', alpha=0.8, density=True)
    ax2.axvline(0, color='red', lw=1)
    ax2.set_xlabel('Residual (pred − actual)')
    ax2.set_ylabel('Density')
    ax2.set_title('Residual distribution')

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig9_pred_vs_actual.pdf')
    plt.show()
    print('Saved: fig9_pred_vs_actual.pdf')

except FileNotFoundError as e:
    print(f'Production model not found: {e}\nRun `make model` first.')

## Cell 5 — Figure: feature importance (winning model)

In [ ]:
if not _FI.exists():
    print('Feature importance not found. Run `make model` first.')
else:
    fi_df = pd.read_csv(_FI)
    rationale = pd.read_csv(_CMP)
    from src.modeling.choose import choose_model
    winner_info = choose_model(rationale)
    winner_name = winner_info['selected_algorithm']

    winner_fi = fi_df[fi_df['algorithm'] == winner_name].head(15).copy()
    if winner_fi.empty:
        print(f'No feature importance for {winner_name}.')
    else:
        def feat_color(name):
            if 'jobtype' in name or 'priority' in name: return '#E69F00'  # categorical
            if 'server_target' in name or 'model_target' in name: return '#009E73'  # target-enc
            return '#0072B2'  # numeric

        colors = [feat_color(f) for f in winner_fi['feature']]
        fig, ax = plt.subplots(figsize=(5.0, 4.0))
        ax.barh(winner_fi['feature'], winner_fi['importance'], color=colors)
        ax.invert_yaxis()
        ax.set_xlabel('Importance (gain)')
        ax.set_title(f'Top 15 features — {winner_name}\n'
                     '(blue=numeric, orange=categorical, green=target-enc)')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'fig10_feature_importance.pdf')
        plt.show()
        print('Saved: fig10_feature_importance.pdf')

## Cell 6 — Figure: prediction latency bar chart

In [ ]:
cmp_df = pd.read_csv(_CMP)
fig, ax = plt.subplots(figsize=(5.0, 2.8))
colors = ['#0072B2'] * len(cmp_df)
bars = ax.bar(cmp_df['algorithm'], cmp_df['latency_ms_10k'], color=colors, alpha=0.85)
ax.axhline(100, color='#D55E00', lw=1.2, ls='--', label='Real-time threshold (100ms)')
ax.set_yscale('log')
ax.set_ylabel('Latency (ms / 10K rows, log scale)')
ax.set_title('Prediction latency by algorithm')
ax.legend(fontsize=6)
plt.xticks(rotation=15, ha='right', fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig11_latency.pdf')
plt.show()
print('Saved: fig11_latency.pdf')

## Cell 7 — Chosen model rationale

In [ ]:
if _RAT.exists():
    with open(_RAT) as f:
        display(Markdown(f.read()))
else:
    print('Rationale not found. Run `make model` first.')

## Cell 8 — Write phase45_headline.json + course report summary

In [ ]:
cmp_df = pd.read_csv(_CMP)
from src.modeling.choose import choose_model
winner_info = choose_model(cmp_df)

linear_rmse = float(cmp_df[cmp_df['algorithm'] == 'LinearRegression']['rmse'].values[0])
winner_rmse = winner_info['rmse']
improvement_pct = round((linear_rmse - winner_rmse) / linear_rmse * 100, 1)

# Top 3 features from winner
top3 = []
if _FI.exists():
    fi_df = pd.read_csv(_FI)
    winner_fi = fi_df[fi_df['algorithm'] == winner_info['selected_algorithm']]
    top3 = winner_fi.head(3)['feature'].tolist()

# LSTM vs LightGBM gap
lstm_rmse = float(cmp_df[cmp_df['algorithm'] == 'LSTM']['rmse'].values[0]) if 'LSTM' in cmp_df['algorithm'].values else None
lgb_rmse  = float(cmp_df[cmp_df['algorithm'] == 'LightGBM']['rmse'].values[0]) if 'LightGBM' in cmp_df['algorithm'].values else None

headline = {
    'winner_algorithm':            winner_info['selected_algorithm'],
    'winner_rmse':                 round(winner_rmse, 4),
    'winner_r2':                   round(winner_info['r2'], 4),
    'winner_latency_ms_per_10k':   round(winner_info['latency_ms_10k'], 2),
    'linear_rmse':                 round(linear_rmse, 4),
    'improvement_over_linear_pct': improvement_pct,
    'top_3_features':              top3,
    'lstm_vs_lightgbm_rmse_gap':   round(lstm_rmse - lgb_rmse, 4) if lstm_rmse and lgb_rmse else None,
    'note_on_lstm':                'LSTM reduces to a 2-layer MLP (seq_len=1); '
                                   'no temporal structure in job_summary features.'
}

out_json = TABLE_DIR / 'phase45_headline.json'
with open(out_json, 'w') as f:
    json.dump(headline, f, indent=2)

print('=== Phase 4.5 Headline ===')
print(json.dumps(headline, indent=2))
print(f'\nSaved: {out_json}')

print(f"""
---
## Course Report Summary

Dataset   : 40M GPU workload records from Alibaba ASI cluster (v2026)
Task      : Predict gpu_hours_requested per job (regression)
Best model: {headline['winner_algorithm']} (RMSE={headline['winner_rmse']}, R²={headline['winner_r2']}, latency={headline['winner_latency_ms_per_10k']}ms/10K)
Baseline  : LinearRegression RMSE={headline['linear_rmse']}
Improvement: {headline['improvement_over_linear_pct']}% RMSE reduction over linear baseline
Top feature: {top3[0] if top3 else 'N/A'} (most predictive of GPU-hours consumed)
LSTM note : {headline['note_on_lstm']}
Limitation: Without pod_hourly data, features are job-level only; adding hourly
             utilization time series in Phase 3b will allow lag-based features
             that may improve accuracy further.
---
""")